# LeetCode 990 — Satisfiability of Equality Equations bằng Python

Link đề: https://leetcode.com/problems/satisfiability-of-equality-equations/ · Độ khó: **Medium** · Chủ đề: Array, String, Union Find, Graph

Skeleton LeetCode:

```python
class Solution:
    def equationsPossible(self, equations: List[str]) -> bool:
        ...
```

Ghi chú: kết quả là `bool` (đáp án duy nhất). Mỗi phương trình có dạng `"x==y"` hoặc `"x!=y"` với `x`, `y` là chữ cái thường (có thể trùng nhau). Cần quyết định có gán được **số nguyên** cho các biến để thoả **mọi** phương trình hay không. Điểm cốt lõi: các phương trình `==` gom biến thành các **lớp tương đương** (bắc cầu!), rồi mọi phương trình `!=` phải nối hai biến thuộc **hai lớp khác nhau**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có thể gán số nguyên cho các biến chữ cái để thoả mọi phương trình `==` và `!=` không?
2. Nắm công cụ: **Union-Find** (DSU) với nén đường đi, duyệt đồ thị (DFS) để tìm thành phần liên thông, "quick-find" bằng mảng nhãn, bao đóng bắc cầu bằng bitmask.
3. Đi từ **thử mọi cách gán số** → **DSU (hai lượt: gộp `==`, rồi kiểm `!=`)** → **thành phần liên thông bằng DFS** → **nhãn quick-find** → **bao đóng bắc cầu Warshall bằng bitmask**.
4. Hiểu mấu chốt: `==` là quan hệ **tương đương** (phản xạ, đối xứng, **bắc cầu**); phải **xử lý hết mọi `==` trước** rồi mới kiểm `!=`; `"a!=a"` luôn sai.
5. Tránh lỗi kinh điển: xử lý theo thứ tự đã cho (một lượt) nên bỏ sót `!=` xuất hiện **trước** `==`, quên tính bắc cầu (`a==b`, `b==c`, `a!=c`), quên `"x!=x"`, nối trực tiếp hai biến thay vì hai gốc trong DSU, dùng sai chỉ số ký tự (`eq[2]` thay vì `eq[3]`).

## 2. Đề bài & ví dụ

Cho mảng chuỗi `equations`, mỗi chuỗi độ dài `4` có dạng `"xi==yi"` hoặc `"xi!=yi"`, trong đó `xi`, `yi` là chữ cái thường (có thể bằng nhau) biểu diễn tên biến.

Trả về `true` nếu có thể **gán số nguyên** cho các biến để thoả **tất cả** phương trình, ngược lại `false`.

**Ví dụ 1:** `equations = ["a==b","b!=a"]` → `false` (nếu `a == b` thì `b != a` sai).

**Ví dụ 2:** `equations = ["b==a","a==b"]` → `true` (gán `a = b = 1`).

**Ràng buộc:** `1 <= equations.length <= 500`; `equations[i].length == 4`; `equations[i][0]` là chữ thường; `equations[i][1]` là `'='` hoặc `'!'`; `equations[i][2]` là `'='`; `equations[i][3]` là chữ thường.

In [1]:
import itertools
import random
import time
from collections import defaultdict, deque
from typing import List

examples = [(["a==b", "b!=a"], False), (["b==a", "a==b"], True)]
for eqs, want in examples:
    print(f"equations = {eqs} -> {want}")

equations = ['a==b', 'b!=a'] -> False
equations = ['b==a', 'a==b'] -> True


## 3. Kiến thức nền

### 3.1 Cú pháp một phương trình

`eq = "x==y"` hoặc `"x!=y"`: `eq[0]` và `eq[3]` là hai biến; `eq[1]` là `'='` (bằng) hoặc `'!'` (khác); `eq[2]` luôn là `'='`. Chỉ số các biến **không** liên tiếp: `eq[0]` và `eq[3]`.

In [2]:
def parse(eq):
    return eq[0], eq[1] == "=", eq[3]          # (biến trái, là '==', biến phải)


print(parse("a==b"), parse("c!=d"), parse("x!=x"))

('a', True, 'b') ('c', False, 'd') ('x', False, 'x')


### 3.2 `==` là quan hệ tương đương

Nếu `a == b` và `b == c` thì `a == c` (bắc cầu); `a == b` ⇒ `b == a` (đối xứng); `a == a` (phản xạ). Do đó các biến tự chia thành **lớp tương đương**. Mọi phương trình `==` cho hai biến cùng lớp; thoả mãn hệ khi và chỉ khi **không có `!=` nào nối hai biến cùng lớp** (mục 10 chứng minh).

### 3.3 Union-Find (DSU)

Mỗi biến là một nút; `find(x)` trả về **gốc** của lớp; `union(x, y)` gộp hai lớp. Nén đường đi (path compression) làm `find` gần `O(1)` đã khấu hao. Với 26 chữ cái, mảng `parent` kích thước 26 là đủ.

In [3]:
def make_dsu(n=26):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]      # nén đường đi kiểu "halving"
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[rx] = ry

    return parent, find, union


parent, find, union = make_dsu()
union(0, 1)
union(1, 2)
print("find(0) == find(2):", find(0) == find(2), "| find(0) == find(3):", find(0) == find(3))

find(0) == find(2): True | find(0) == find(3): False


## 4. Ý tưởng nền tảng / trực giác

Xét `["a==b", "b==c", "a!=c"]`. Từ hai phương trình `==`, `a`, `b`, `c` cùng lớp; nhưng `a != c` đòi chúng khác nhau ⇒ **mâu thuẫn** (`false`). Nếu ta chỉ nhìn trực tiếp từng cặp (không bắc cầu) thì `a` và `c` không xuất hiện trong phương trình `==` nào ⇒ tưởng ổn: sai.

Một thứ tự khác: `["a!=c", "a==b", "b==c"]`. Nếu xử lý lần lượt và kiểm `!=` ngay thì lúc đó `a`, `c` chưa cùng lớp ⇒ ổn; về sau `==` mới nối chúng ⇒ mâu thuẫn bị bỏ sót. **Giải pháp:** hai lượt — gộp **mọi** `==` trước, rồi kiểm các `!=`.

In [4]:
def demo_two_pass(eqs):
    parent, find, union = make_dsu()
    for eq in eqs:
        if eq[1] == "=":
            union(ord(eq[0]) - 97, ord(eq[3]) - 97)
    return all(find(ord(e[0]) - 97) != find(ord(e[3]) - 97) for e in eqs if e[1] == "!")


print(demo_two_pass(["a==b", "b==c", "a!=c"]), demo_two_pass(["a!=c", "a==b", "b==c"]), demo_two_pass(["a==b", "b!=c"]))

False False True


## 5. Brute force — thử mọi cách gán số

**Ý tưởng:** gọi `m` là số biến phân biệt xuất hiện. Chỉ **mẫu bằng nhau** giữa các biến quan trọng, nên mỗi biến chỉ cần nhận giá trị trong `{0, …, m−1}` (mọi phân hoạch của `m` biến thành các lớp cần tối đa `m` nhãn). Thử mọi `m^m` cách gán, kiểm tra từng phương trình. Với `m ≤ 7` là vài trăm nghìn cách — đủ cho **oracle** theo đúng định nghĩa.

In [5]:
def possible_brute(equations):
    vars_ = sorted({c for e in equations for c in (e[0], e[3])})
    m = len(vars_)
    for values in itertools.product(range(m), repeat=m):
        val = dict(zip(vars_, values))
        if all((val[e[0]] == val[e[3]]) == (e[1] == "=") for e in equations):
            return True
    return False


for eqs, want in examples:
    assert possible_brute(eqs) == want
print([possible_brute(e) for e, _ in examples])

[False, True]


In [6]:
# Code nộp LeetCode cho Brute force (chỉ minh hoạ: m^m cách gán, bất khả thi với 26 biến)
import itertools


class SolutionBrute:
    def equationsPossible(self, equations: List[str]) -> bool:
        vars_ = sorted({c for e in equations for c in (e[0], e[3])})
        m = len(vars_)
        for values in itertools.product(range(m), repeat=m):
            val = dict(zip(vars_, values))
            if all((val[e[0]] == val[e[3]]) == (e[1] == "=") for e in equations):
                return True
        return False

**Độ phức tạp:** `O(m^m · E)`; với `m = 26` bất khả thi.

**Nút thắt:** ta thử cả `m^m` cách gán trong khi chỉ cần biết biến nào **bị ép bằng nhau**. Đó là bao đóng bắc cầu của các phương trình `==` — một bài toán đồ thị.

## 6. Bước đột phá

Coi mỗi biến là một đỉnh; mỗi `==` là một cạnh. Các **thành phần liên thông** chính là các lớp tương đương. Hệ thoả mãn ⇔ mọi `!=` nối hai đỉnh thuộc **hai thành phần khác nhau**. Dùng Union-Find: gộp mọi `==`, rồi kiểm `!=`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** Union-Find (đáp án chính), **Cách 3** thành phần liên thông bằng DFS, **Cách 4** nhãn quick-find, **Cách 5** bao đóng bắc cầu bằng bitmask.

### Cách 2 — Union-Find hai lượt

**Giải thích từng đoạn:**

1. `parent[i] = i` cho 26 chữ cái.
2. **Lượt 1:** với mỗi `x==y`, `union(x, y)`.
3. **Lượt 2:** với mỗi `x!=y`, nếu `find(x) == find(y)` trả `False`.
4. Không vi phạm ⇒ `True`.

In [7]:
def possible_dsu(equations):
    parent = list(range(26))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for e in equations:                                  # lượt 1: gộp mọi ==
        if e[1] == "=":
            parent[find(ord(e[0]) - 97)] = find(ord(e[3]) - 97)
    for e in equations:                                  # lượt 2: kiểm mọi !=
        if e[1] == "!" and find(ord(e[0]) - 97) == find(ord(e[3]) - 97):
            return False
    return True


for eqs, want in examples:
    assert possible_dsu(eqs) == want
print([possible_dsu(e) for e, _ in examples])

[False, True]


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionDSU:
    def equationsPossible(self, equations: List[str]) -> bool:
        parent = list(range(26))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        for e in equations:
            if e[1] == "=":
                parent[find(ord(e[0]) - 97)] = find(ord(e[3]) - 97)
        for e in equations:
            if e[1] == "!" and find(ord(e[0]) - 97) == find(ord(e[3]) - 97):
                return False
        return True

**Độ phức tạp:** `O(E · α(26))` ≈ `O(E)` thời gian, `O(1)` bộ nhớ (26 phần tử).

### Cách 3 — Thành phần liên thông bằng DFS

**Ý tưởng:** dựng đồ thị vô hướng 26 đỉnh với cạnh cho mỗi `==`; duyệt DFS gán **nhãn thành phần** `comp[v]`; rồi kiểm mọi `!=`: `comp[x] != comp[y]`.

In [9]:
def possible_dfs(equations):
    adj = defaultdict(list)
    for e in equations:
        if e[1] == "=":
            adj[e[0]].append(e[3])
            adj[e[3]].append(e[0])
    comp = {}
    for start in map(chr, range(97, 123)):
        if start in comp:
            continue
        stack = [start]
        comp[start] = start
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if v not in comp:
                    comp[v] = start
                    stack.append(v)
    return all(comp[e[0]] != comp[e[3]] for e in equations if e[1] == "!")


for eqs, want in examples:
    assert possible_dfs(eqs) == want
print([possible_dfs(e) for e, _ in examples])

[False, True]


In [10]:
# Code nộp LeetCode cho Cách 3
from collections import defaultdict


class SolutionDFS:
    def equationsPossible(self, equations: List[str]) -> bool:
        adj = defaultdict(list)
        for e in equations:
            if e[1] == "=":
                adj[e[0]].append(e[3])
                adj[e[3]].append(e[0])
        comp = {}
        for start in "abcdefghijklmnopqrstuvwxyz":
            if start in comp:
                continue
            comp[start] = start
            stack = [start]
            while stack:
                u = stack.pop()
                for v in adj[u]:
                    if v not in comp:
                        comp[v] = start
                        stack.append(v)
        return all(comp[e[0]] != comp[e[3]] for e in equations if e[1] == "!")

**Độ phức tạp:** `O(26 + E)` thời gian, `O(26 + E)` bộ nhớ.

### Cách 4 — Nhãn quick-find

**Ý tưởng:** giữ mảng `label[26]`: biến cùng lớp có cùng nhãn. Gộp `x==y`: nếu nhãn khác nhau, đổi **mọi** biến có nhãn của `x` sang nhãn của `y` (`O(26)` mỗi lần gộp). Kiểm `!=` bằng so sánh nhãn.

In [11]:
def possible_quickfind(equations):
    label = list(range(26))
    for e in equations:
        if e[1] == "=":
            a, b = label[ord(e[0]) - 97], label[ord(e[3]) - 97]
            if a != b:
                label = [b if t == a else t for t in label]      # gộp hai lớp: đổi nhãn a thành b
    return all(label[ord(e[0]) - 97] != label[ord(e[3]) - 97] for e in equations if e[1] == "!")


for eqs, want in examples:
    assert possible_quickfind(eqs) == want
print([possible_quickfind(e) for e, _ in examples])

[False, True]


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionQuickFind:
    def equationsPossible(self, equations: List[str]) -> bool:
        label = list(range(26))
        for e in equations:
            if e[1] == "=":
                a, b = label[ord(e[0]) - 97], label[ord(e[3]) - 97]
                if a != b:
                    label = [b if t == a else t for t in label]
        return all(label[ord(e[0]) - 97] != label[ord(e[3]) - 97] for e in equations if e[1] == "!")

**Độ phức tạp:** `O(26 · E)` thời gian (mỗi lần gộp quét 26 phần tử), `O(26)` bộ nhớ — vẫn nhỏ vì chỉ có 26 biến.

### Cách 5 — Bao đóng bắc cầu bằng bitmask (Warshall)

**Ý tưởng:** `reach[i]` là bitmask các biến bằng `i` (khởi tạo `1 << i`, cộng các cạnh `==` hai chiều). Bao đóng bắc cầu Warshall: với mỗi `k`, với mỗi `i` có `k ∈ reach[i]` thì `reach[i] |= reach[k]`. Sau đó `!=` giữa `x, y` hợp lệ ⇔ `y ∉ reach[x]`.

In [13]:
def possible_closure(equations):
    reach = [1 << i for i in range(26)]
    for e in equations:
        if e[1] == "=":
            a, b = ord(e[0]) - 97, ord(e[3]) - 97
            reach[a] |= 1 << b
            reach[b] |= 1 << a
    for k in range(26):                                  # Warshall: đi qua đỉnh trung gian k
        for i in range(26):
            if reach[i] >> k & 1:
                reach[i] |= reach[k]
    return all(not (reach[ord(e[0]) - 97] >> (ord(e[3]) - 97) & 1) for e in equations if e[1] == "!")


for eqs, want in examples:
    assert possible_closure(eqs) == want
print([possible_closure(e) for e, _ in examples])

[False, True]


In [14]:
# Code nộp LeetCode cho Cách 5
class SolutionClosure:
    def equationsPossible(self, equations: List[str]) -> bool:
        reach = [1 << i for i in range(26)]
        for e in equations:
            if e[1] == "=":
                a, b = ord(e[0]) - 97, ord(e[3]) - 97
                reach[a] |= 1 << b
                reach[b] |= 1 << a
        for k in range(26):
            for i in range(26):
                if reach[i] >> k & 1:
                    reach[i] |= reach[k]
        return all(not (reach[ord(e[0]) - 97] >> (ord(e[3]) - 97) & 1) for e in equations if e[1] == "!")

**Độ phức tạp:** `O(26² + E)` thời gian (hằng số lớn hơn nhưng vẫn hằng số), `O(26)` bộ nhớ.

## 8. TRACE từng bước

**Trace Cách 2** với `["a==b", "b==c", "a!=c", "d==e"]`:

| bước | phương trình | thao tác | lớp (nhóm) sau bước |
|---|---|---|---|
| lượt 1 | `a==b` | `union(a, b)` | `{a,b}` |
| lượt 1 | `b==c` | `union(b, c)` | `{a,b,c}` |
| lượt 1 | `d==e` | `union(d, e)` | `{a,b,c}`, `{d,e}` |
| lượt 2 | `a!=c` | `find(a) == find(c)` ⇒ **mâu thuẫn** | `False` |

Hàm trace in từng bước (kèm các nhóm hiện tại):

In [15]:
def trace_dsu(equations):
    parent = list(range(26))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def groups():
        g = defaultdict(list)
        for i in range(26):
            g[find(i)].append(chr(97 + i))
        return sorted([v for v in g.values() if len(v) > 1])

    print("equations =", equations)
    for e in equations:
        if e[1] == "=":
            parent[find(ord(e[0]) - 97)] = find(ord(e[3]) - 97)
            print(f"  lượt 1: {e}: union -> nhóm {groups()}")
    for e in equations:
        if e[1] == "!":
            same = find(ord(e[0]) - 97) == find(ord(e[3]) - 97)
            print(f"  lượt 2: {e}: find({e[0]}) {'==' if same else '!='} find({e[3]}) -> {'MÂU THUẪN' if same else 'ổn'}")
            if same:
                print("=> False")
                return False
    print("=> True")
    return True


assert trace_dsu(["a==b", "b==c", "a!=c", "d==e"]) is False
assert trace_dsu(["a==b", "b!=c", "d==e"]) is True

equations = ['a==b', 'b==c', 'a!=c', 'd==e']
  lượt 1: a==b: union -> nhóm [['a', 'b']]
  lượt 1: b==c: union -> nhóm [['a', 'b', 'c']]
  lượt 1: d==e: union -> nhóm [['a', 'b', 'c'], ['d', 'e']]
  lượt 2: a!=c: find(a) == find(c) -> MÂU THUẪN
=> False
equations = ['a==b', 'b!=c', 'd==e']
  lượt 1: a==b: union -> nhóm [['a', 'b']]
  lượt 1: d==e: union -> nhóm [['a', 'b'], ['d', 'e']]
  lượt 2: b!=c: find(b) != find(c) -> ổn
=> True


**Trace thứ tự:** `["a!=c", "a==b", "b==c"]` — cách một lượt (kiểm `!=` ngay) sẽ trả `True` sai; hai lượt trả `False` đúng (xem mục 13).

## 9. Code nộp LeetCode

Chọn **Cách 2** (Union-Find hai lượt). Giữ nguyên `class Solution`, `equationsPossible(self, equations)`:

In [16]:
class Solution:
    def equationsPossible(self, equations: List[str]) -> bool:
        parent = list(range(26))                          # mỗi chữ cái là một lớp riêng

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]             # nén đường đi
                x = parent[x]
            return x

        for e in equations:                               # lượt 1: gộp mọi "=="
            if e[1] == "=":
                parent[find(ord(e[0]) - 97)] = find(ord(e[3]) - 97)
        for e in equations:                               # lượt 2: mọi "!=" phải nối hai lớp khác nhau
            if e[1] == "!" and find(ord(e[0]) - 97) == find(ord(e[3]) - 97):
                return False
        return True


print([Solution().equationsPossible(e) for e, _ in examples])
assert all(Solution().equationsPossible(e) == w for e, w in examples)

[False, True]


## 10. Vì sao đúng

**(a) Hệ thoả mãn được ⇔ không có `!=` nào nối hai biến cùng lớp tương đương.** Gọi `~` là bao đóng phản xạ–đối xứng–bắc cầu của các cạnh `==`.

- (⇒) Nếu có phép gán thoả mãn thì mọi cạnh `==` cho hai biến cùng giá trị; bởi tính bắc cầu của "cùng giá trị", mọi cặp `x ~ y` có cùng giá trị. Nếu có `x != y` với `x ~ y` thì `val(x) = val(y)` nhưng phương trình đòi `val(x) ≠ val(y)`: mâu thuẫn.
- (⇐) Nếu không có `!=` nào nối hai biến cùng lớp, gán mỗi **lớp** một số nguyên **riêng biệt** (ví dụ chỉ số lớp). Mọi `==` nối hai biến cùng lớp ⇒ cùng giá trị ✓. Mọi `!=` nối hai biến khác lớp ⇒ giá trị khác ✓. Gán thoả mãn.

**(b) Hệ quả cho `"x!=x"`:** `x ~ x` luôn đúng (phản xạ) nên `"x!=x"` luôn vi phạm ⇒ `False`. Mọi cài đặt đúng tự xử lý vì `find(x) == find(x)`.

**(c) DSU đúng.** Union-Find duy trì đúng các lớp tương đương của các `==` đã gộp (find cho đại diện duy nhất của lớp). Việc gộp **mọi** `==` trước rồi mới kiểm `!=` đảm bảo các lớp cuối cùng đã đầy đủ; kiểm `!=` trong lúc gộp có thể bỏ sót vì hai biến có thể chỉ bị nối ở các `==` sau.

**(d) Cách 3.** Thành phần liên thông của đồ thị các `==` chính là lớp bắc cầu (DFS gán cùng nhãn cho mọi đỉnh tới được).

**(e) Cách 4.** Bất biến: `label[i] == label[j] ⇔ i ~ j` theo các `==` đã xử lý. Gộp `x==y` đổi mọi nhãn của lớp `x` thành nhãn của lớp `y`, giữ bất biến.

**(f) Cách 5.** `reach[i]` sau Warshall là tập các `j` với `i ~ j`: bao đóng bắc cầu của quan hệ cạnh hai chiều (Warshall chuẩn: sau khi xét `k`, `reach` chứa mọi đường đi chỉ qua các đỉnh trung gian `≤ k`).

**(g) Brute force đúng.** Chỉ **mẫu bằng nhau** của các biến quan trọng (mỗi phương trình chỉ hỏi `==` hay `≠`). Mọi phân hoạch `m` biến thành lớp có thể biểu diễn bằng gán giá trị trong `{0..m−1}`; do đó tồn tại gán thoả mãn ⇔ tồn tại gán thoả mãn với giá trị trong `{0..m−1}`. Ta kiểm chứng thêm bằng máy với `{0..m+1}` ở mục 12.

**(h) Kiểm chứng bằng máy.** Với **mọi** hệ phương trình trên 3 biến `{a, b, c}` (`2^18 = 262 144` tập con của 18 phương trình, kể cả `x==x`, `x!=x`), oracle theo định nghĩa (tồn tại phép gán `{0,1,2}`) khớp mọi cách; mẫu lớn hơn với 4–6 biến khớp brute; hệ lớn (500 phương trình, 26 biến) khớp nhau và khớp các tính chất dựng sẵn (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi phép gán | `O(m^m · E)` | `O(m)` | oracle `m ≲ 7` |
| 2. Union-Find hai lượt | `O(E α(26))` | `O(1)` | **đáp án chính** |
| 3. DFS thành phần | `O(26 + E)` | `O(26 + E)` | tường minh về đồ thị |
| 4. Quick-find nhãn | `O(26·E)` | `O(26)` | cài đặt đơn giản nhất |
| 5. Bao đóng bitmask | `O(26² + E)` | `O(26)` | tổng quát cho quan hệ bắc cầu |

Đo thực tế với `E = 500` phương trình:

In [17]:
def random_equations(rng, n, nvars=26, p_eq=0.5):
    letters = [chr(97 + i) for i in range(nvars)]
    return [rng.choice(letters) + ("==" if rng.random() < p_eq else "!=") + rng.choice(letters) for _ in range(n)]


def bench(fn, arg, repeat=100):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(arg)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(990)
big = random_equations(rng, 500, 26, 0.9)
for name, fn in [("dsu", possible_dsu), ("dfs", possible_dfs), ("quickfind", possible_quickfind), ("closure", possible_closure)]:
    print(f"{name:10s}: {bench(fn, big) * 1e6:8.1f} us (E = 500)")

dsu       :    109.7 us (E = 500)
dfs       :     71.9 us (E = 500)
quickfind :     56.4 us (E = 500)


closure   :    136.1 us (E = 500)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi hệ trên 3 biến, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "dsu": possible_dsu,
    "dfs": possible_dfs,
    "quickfind": possible_quickfind,
    "closure": possible_closure,
    "brute": possible_brute,
    "Solution": lambda e: Solution().equationsPossible(list(e)),
    "SolutionDSU": lambda e: SolutionDSU().equationsPossible(list(e)),
    "SolutionDFS": lambda e: SolutionDFS().equationsPossible(list(e)),
    "SolutionQuickFind": lambda e: SolutionQuickFind().equationsPossible(list(e)),
    "SolutionClosure": lambda e: SolutionClosure().equationsPossible(list(e)),
    "SolutionBrute": lambda e: SolutionBrute().equationsPossible(list(e)),
}
SLOW = {"brute", "SolutionBrute"}
fast = [n for n in IMPLS if n not in SLOW]


def check(eqs, want=None, names=None):
    nvars = len({c for e in eqs for c in (e[0], e[3])})
    if want is None:
        want = possible_brute(eqs) if nvars <= 7 else None
    results = {}
    for name in (names or IMPLS):
        if name in SLOW and nvars > 7:
            continue
        results[name] = IMPLS[name](eqs)
    assert len(set(results.values())) == 1, (eqs, results)
    got = next(iter(results.values()))
    if want is not None:
        assert got == want, (eqs, got, want)
    return got


for eqs, want in examples:
    check(eqs, want)
print("ví dụ đề: oracle brute và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle brute và 11 cài đặt đúng


### B. Edge case

- Một phương trình: `"a==a"` (đúng), `"a!=a"` (**sai**), `"a==b"`, `"a!=b"`.
- Thứ tự: `!=` đứng **trước** `==` gây mâu thuẫn.
- Bắc cầu dài: `a==b, b==c, c==d, a!=d`.
- Đồ thị `==` có chu trình (không ảnh hưởng).
- Phương trình trùng lặp.
- Một biến bằng chính nó cùng `!=` với biến khác.
- Mọi `!=` giữa các biến đôi một khác lớp (đúng).
- 26 biến nối vòng `a==b … y==z`, rồi `a!=z` (**sai**).

In [19]:
chain26 = [chr(97 + i) + "==" + chr(98 + i) for i in range(25)]
edge = [
    ["a==a"], ["a!=a"], ["a==b"], ["a!=b"], ["a!=c", "a==b", "b==c"], ["a==b", "b==c", "c==d", "a!=d"], ["a==b", "b==c", "c==a"],
    ["a==b", "a==b", "b==a"], ["a==a", "a!=b"], ["a!=b", "b!=c", "c!=a"], ["a==b", "b!=c", "c==d", "d!=a"], ["a!=b", "b==c", "a==c"],
    chain26 + ["a!=z"], chain26 + ["z!=a"], chain26 + ["a==z"], ["a==b", "c==d", "b!=c", "d==a"], ["x==y", "y==z", "x!=z", "p==q"],
]
for eqs in edge:
    check(eqs)
print(len(edge), "edge case: mọi cài đặt khớp (brute cho ≤ 7 biến)")
assert Solution().equationsPossible(["a!=a"]) is False and Solution().equationsPossible(chain26 + ["a!=z"]) is False
assert Solution().equationsPossible(["a!=c", "a==b", "b==c"]) is False

17 edge case: mọi cài đặt khớp (brute cho ≤ 7 biến)


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_one_pass(equations):                           # xử lý theo thứ tự, kiểm != ngay
    parent = list(range(26))

    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x

    for e in equations:
        a, b = ord(e[0]) - 97, ord(e[3]) - 97
        if e[1] == "=":
            parent[find(a)] = find(b)
        elif find(a) == find(b):
            return False
    return True


def bug_no_transitive(equations):                      # chỉ kiểm cặp trực tiếp
    eq_pairs = {frozenset((e[0], e[3])) for e in equations if e[1] == "="}
    return all(frozenset((e[0], e[3])) not in eq_pairs for e in equations if e[1] == "!")


def bug_skip_self_neq(equations):                      # bỏ qua "x!=x"
    parent = list(range(26))

    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x

    for e in equations:
        if e[1] == "=":
            parent[find(ord(e[0]) - 97)] = find(ord(e[3]) - 97)
    return all(find(ord(e[0]) - 97) != find(ord(e[3]) - 97) for e in equations if e[1] == "!" and e[0] != e[3])


def bug_union_no_find(equations):                      # nối trực tiếp hai biến, không qua gốc
    parent = list(range(26))

    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x

    for e in equations:
        if e[1] == "=":
            parent[ord(e[0]) - 97] = ord(e[3]) - 97     # BUG: không nối gốc với gốc
    return all(find(ord(e[0]) - 97) != find(ord(e[3]) - 97) for e in equations if e[1] == "!")


def bug_wrong_index(equations):                        # dùng eq[2] ('=') thay vì eq[3]
    parent = list(range(130))

    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x

    for e in equations:
        if e[1] == "=":
            parent[find(ord(e[0]))] = find(ord(e[2]))
    return all(find(ord(e[0])) != find(ord(e[2])) for e in equations if e[1] == "!")


def bug_neq_means_union(equations):                    # hiểu nhầm: != cũng gộp hai biến rồi kiểm lại (trả sai)
    parent = list(range(26))

    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x

    for e in equations:
        if e[1] == "=":
            parent[find(ord(e[0]) - 97)] = find(ord(e[3]) - 97)
    return not any(find(ord(e[0]) - 97) == find(ord(e[3]) - 97) for e in equations if e[1] == "!") and len(equations) > 1


def bug_always_true_without_eq(equations):             # chỉ nhìn các '==' kề nhau (không DSU)
    adj = defaultdict(set)
    for e in equations:
        if e[1] == "=":
            adj[e[0]].add(e[3])
            adj[e[3]].add(e[0])
    return all(e[3] not in adj[e[0]] and e[0] != e[3] for e in equations if e[1] == "!")


BUGS = [bug_one_pass, bug_no_transitive, bug_skip_self_neq, bug_union_no_find, bug_wrong_index, bug_neq_means_union, bug_always_true_without_eq]
probe = [["a!=c", "a==b", "b==c"], ["a==b", "b==c", "a!=c"], ["a!=a"], ["a==b", "b==c", "c==d", "a!=d"], ["a==b", "b==c", "c!=a"], ["a==b"], ["a!=b"],
         ["a==b", "c==d", "b!=c", "d==a"], ["a!=b", "b==a"], ["a==b", "b!=c", "d==e"], ["a==b", "a==c", "b!=c"]]
for bug in BUGS:
    found = None
    for eqs in probe:
        try:
            got = bug(eqs)
        except IndexError as e:
            got = "IndexError"
        if got != possible_brute(eqs):
            found = (eqs, got)
            break
    assert found, bug.__name__
    print(f"{bug.__name__:28s} sai trên {found[0]}: ra {found[1]}, đúng {possible_brute(found[0])}")

bug_one_pass                 sai trên ['a!=c', 'a==b', 'b==c']: ra True, đúng False
bug_no_transitive            sai trên ['a!=c', 'a==b', 'b==c']: ra True, đúng False
bug_skip_self_neq            sai trên ['a!=a']: ra True, đúng False
bug_union_no_find            sai trên ['a==b', 'a==c', 'b!=c']: ra True, đúng False
bug_wrong_index              sai trên ['a!=a']: ra True, đúng False
bug_neq_means_union          sai trên ['a==b']: ra False, đúng True
bug_always_true_without_eq   sai trên ['a!=c', 'a==b', 'b==c']: ra True, đúng False


### D. Vét cạn mọi hệ trên 3 biến `{a, b, c}`

Có `3 × 3 × 2 = 18` phương trình (kể cả `a==a`, `a!=a`). Mỗi hệ là một **tập con** của 18 phương trình (thứ tự và trùng lặp không ảnh hưởng đáp án): `2^18 = 262 144` hệ. Oracle theo **định nghĩa**: với mỗi phép gán `(va, vb, vc) ∈ {0,1,2}^3` ta tính mặt nạ các phương trình được thoả; hệ `S` thoả mãn được ⇔ tồn tại phép gán có mặt nạ chứa `S`. Hệ rỗng không thuộc phạm vi đề (`≥ 1` phương trình) nên bỏ.

In [21]:
L3 = "abc"
EQ3 = [x + op + y for x in L3 for y in L3 for op in ("==", "!=")]
assert len(EQ3) == 18
masks = set()
for va, vb, vc in itertools.product(range(3), repeat=3):
    val = {"a": va, "b": vb, "c": vc}
    m = 0
    for i, e in enumerate(EQ3):
        if (val[e[0]] == val[e[3]]) == (e[1] == "="):
            m |= 1 << i
    masks.add(m)
masks = sorted(masks)
print("số mặt nạ phân biệt (phép gán):", len(masks))

t0 = time.perf_counter()
main_impls = ["dsu", "dfs", "quickfind", "closure", "Solution"]
cnt = true_cnt = 0
for subset in range(1, 1 << 18):
    want = any(subset & ~m == 0 for m in masks)
    eqs = [EQ3[i] for i in range(18) if subset >> i & 1]
    for name in main_impls:
        assert IMPLS[name](eqs) == want, (name, eqs, want)
    cnt += 1
    true_cnt += want
print(cnt, "hệ (mọi tập con khác rỗng của 18 phương trình):", len(main_impls), "cài đặt khớp oracle định nghĩa; hệ thoả mãn được:", true_cnt, ";", round(time.perf_counter() - t0, 1), "s")
assert cnt == 2 ** 18 - 1

số mặt nạ phân biệt (phép gán): 5


262143 hệ (mọi tập con khác rỗng của 18 phương trình): 5 cài đặt khớp oracle định nghĩa; hệ thoả mãn được: 2095 ; 13.9 s


In [22]:
# các class nộp còn lại + brute trên mẫu cố định của 262 143 hệ; kiểm chứng oracle giá trị {0..m-1} so với {0..m+1}
rng = random.Random(99001)
others = ["SolutionDSU", "SolutionDFS", "SolutionQuickFind", "SolutionClosure", "SolutionBrute", "brute"]
for _ in range(3000):
    subset = rng.randrange(1, 1 << 18)
    eqs = [EQ3[i] for i in range(18) if subset >> i & 1]
    want = any(subset & ~m == 0 for m in masks)
    for name in others:
        assert IMPLS[name](eqs) == want, (name, eqs)


def possible_brute_wider(equations, extra=2):
    vars_ = sorted({c for e in equations for c in (e[0], e[3])})
    m = len(vars_)
    for values in itertools.product(range(m + extra), repeat=m):
        val = dict(zip(vars_, values))
        if all((val[e[0]] == val[e[3]]) == (e[1] == "=") for e in equations):
            return True
    return False


for _ in range(1500):
    eqs = random_equations(rng, rng.randint(1, 8), 4, 0.5)
    assert possible_brute(eqs) == possible_brute_wider(eqs)
print("3000 hệ mẫu: các class còn lại và brute khớp; 1500 hệ 4 biến: giá trị {0..m-1} và {0..m+1} cho cùng đáp án")

3000 hệ mẫu: các class còn lại và brute khớp; 1500 hệ 4 biến: giá trị {0..m-1} và {0..m+1} cho cùng đáp án


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) **Dựng thuận**: chọn một phép gán ngẫu nhiên `val: letters → {0..k}`, sinh phương trình nhất quán với `val` (`==` nếu cùng giá trị, `!=` nếu khác) ⇒ đáp án phải là `True`; (2) **thêm mâu thuẫn**: lấy một phương trình `x==y` có sẵn và thêm `y!=x` ⇒ chắc chắn `False`; (3) **không phụ thuộc thứ tự**: xáo trộn; (4) **trùng lặp không đổi đáp án**; (5) khớp brute với `≤ 6` biến; (6) các cài đặt khớp nhau với 26 biến.

In [23]:
def consistent_equations(rng, n, nvars, k):
    val = {chr(97 + i): rng.randint(0, k) for i in range(nvars)}
    letters = list(val)
    eqs = []
    for _ in range(n):
        x, y = rng.choice(letters), rng.choice(letters)
        eqs.append(x + ("==" if val[x] == val[y] else "!=") + y)
    return eqs, val


rng = random.Random(20240990)
t0 = time.perf_counter()
n_false = 0
for it in range(1500):
    nvars = rng.choice([3, 5, 6, 12, 26])
    n = rng.randint(1, 500) if nvars > 6 else rng.randint(1, 12)
    k = rng.choice([1, 2, 5, 25])
    eqs, val = consistent_equations(rng, n, nvars, k)
    for name in fast:
        assert IMPLS[name](eqs) is True, (name, it)                                  # (1)
    sh = eqs[:]
    rng.shuffle(sh)
    assert Solution().equationsPossible(sh) is True                                  # (3)
    assert Solution().equationsPossible(eqs + eqs) is True                           # (4)
    eq_only = [e for e in eqs if e[1] == "="]
    if eq_only:
        e = rng.choice(eq_only)
        bad = eqs + [e[3] + "!=" + e[0]]                                             # đảo một '==' thành '!=' => chắc chắn mâu thuẫn
        rng.shuffle(bad)
        for name in fast:
            assert IMPLS[name](bad) is False, (name, it)                             # (2)
        n_false += 1
    if len({c for e in eqs for c in (e[0], e[3])}) <= 6 and nvars <= 6:
        assert possible_brute(eqs) is True                                           # (5)
print("1500 hệ nhất quán (tới 500 phương trình, 26 biến): 5 tính chất đúng; số hệ đã thêm mâu thuẫn:", n_false, ";", round(time.perf_counter() - t0, 1), "s")

1500 hệ nhất quán (tới 500 phương trình, 26 biến): 5 tính chất đúng; số hệ đã thêm mâu thuẫn: 1357 ; 1.2 s


In [24]:
# hệ hoàn toàn ngẫu nhiên: các cài đặt khớp nhau (26 biến) và khớp brute (≤ 6 biến)
rng = random.Random(99002)
t0 = time.perf_counter()
cnt_true = 0
for _ in range(4000):
    nvars = rng.choice([2, 3, 4, 5, 6, 26])
    eqs = random_equations(rng, rng.randint(1, 30 if nvars <= 6 else 500), nvars, rng.choice([0.3, 0.5, 0.8, 0.95]))
    results = {name: IMPLS[name](eqs) for name in fast}
    assert len(set(results.values())) == 1, (eqs[:5], results)
    cnt_true += results["Solution"]
    if len({c for e in eqs for c in (e[0], e[3])}) <= 6:
        assert possible_brute(eqs) == results["Solution"]
print("4000 hệ ngẫu nhiên: các cài đặt khớp nhau và khớp brute (≤ 6 biến); số hệ True:", cnt_true, ";", round(time.perf_counter() - t0, 1), "s")

4000 hệ ngẫu nhiên: các cài đặt khớp nhau và khớp brute (≤ 6 biến); số hệ True: 757 ; 18.6 s


### F. Giới hạn

`E = 500`, đủ 26 biến; chuỗi dài `a==b, b==c, …` (26 biến) rồi một `!=` ở cuối (mâu thuẫn xa); chuỗi `==` đảo thứ tự (nhiều lần `union` cần nén đường đi); toàn `!=` giữa các biến khác nhau; `500` bản sao của một phương trình.

In [25]:
t0 = time.perf_counter()
chain = [chr(97 + i) + "==" + chr(98 + i) for i in range(25)]
rev_chain = [chr(98 + i) + "==" + chr(97 + i) for i in reversed(range(25))]
cases = [
    (chain * 19 + ["a!=z"], False), (rev_chain * 19 + ["z!=a"], False), (chain * 19 + ["a!=a"], False),
    ([chr(97 + i) + "!=" + chr(97 + (i + 1) % 26) for i in range(26)] * 19, True),
    (["a==b"] * 500, True), (["a!=b"] * 500, True), (["a==b"] * 250 + ["a!=b"] * 250, False),
]
for eqs, want in cases:
    assert len(eqs) <= 500
    for name in fast:
        assert IMPLS[name](eqs) is want, (name, eqs[:3], want)
rng = random.Random(99003)
for _ in range(200):
    eqs = random_equations(rng, 500, 26, rng.choice([0.5, 0.9, 0.99]))
    results = {name: IMPLS[name](eqs) for name in fast}
    assert len(set(results.values())) == 1
print("E = 500, 26 biến: chuỗi dài, đảo thứ tự, trùng lặp, 200 hệ ngẫu nhiên: mọi cách khớp;", round(time.perf_counter() - t0, 2), "s")

E = 500, 26 biến: chuỗi dài, đảo thứ tự, trùng lặp, 200 hệ ngẫu nhiên: mọi cách khớp; 0.23 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: xử lý theo thứ tự đã cho (một lượt)

In [26]:
eqs = ["a!=c", "a==b", "b==c"]
print("bug:", bug_one_pass(eqs), "| đúng:", Solution().equationsPossible(eqs))
assert bug_one_pass(eqs) is True and Solution().equationsPossible(eqs) is False

bug: True | đúng: False


**Vì sao sai:** khi gặp `a!=c` các `==` sau chưa được áp dụng nên `a`, `c` chưa cùng lớp; mâu thuẫn xuất hiện sau đó bị bỏ sót. **Đúng:** gộp **mọi** `==` trước, rồi mới kiểm `!=`.

### Lỗi 2: quên tính bắc cầu

In [27]:
eqs = ["a==b", "b==c", "a!=c"]
print("bug:", bug_no_transitive(eqs), "| đúng:", Solution().equationsPossible(eqs))
assert bug_no_transitive(eqs) is True and Solution().equationsPossible(eqs) is False

bug: True | đúng: False


**Vì sao sai:** `a==b` và `b==c` kéo theo `a==c`. Chỉ nhìn cặp trực tiếp không thấy `a != c` mâu thuẫn. **Đúng:** Union-Find / thành phần liên thông.

### Lỗi 3: quên `"x!=x"`

In [28]:
print("bug:", bug_skip_self_neq(["a!=a"]), "| đúng:", Solution().equationsPossible(["a!=a"]))
assert bug_skip_self_neq(["a!=a"]) is True and Solution().equationsPossible(["a!=a"]) is False

bug: True | đúng: False


**Vì sao sai:** mọi biến bằng chính nó, nên `x != x` không thể thoả. **Đúng:** để `find(x) == find(x)` tự bắt (đừng loại trừ trường hợp này).

### Lỗi 4: nối trực tiếp hai biến thay vì hai gốc

In [29]:
eqs = ["a==b", "a==c", "b!=c"]
print("bug:", bug_union_no_find(eqs), "| đúng:", Solution().equationsPossible(eqs))
assert bug_union_no_find(eqs) is True and Solution().equationsPossible(eqs) is False

bug: True | đúng: False


**Vì sao sai:** gán `parent[x] = y` khi `x` không phải gốc làm **mất** các biến đã gộp trước với `x` (chúng vẫn trỏ về gốc cũ). **Đúng:** `parent[find(x)] = find(y)`.

### Lỗi 5: sai chỉ số ký tự

In [30]:
try:
    r = bug_wrong_index(["a==b", "b!=a"])
    print("bug ra:", r, "| đúng:", Solution().equationsPossible(["a==b", "b!=a"]))
except IndexError as e:
    print("IndexError:", e)
assert bug_wrong_index(["a==b", "b!=a"]) != Solution().equationsPossible(["a==b", "b!=a"])

bug ra: True | đúng: False


**Vì sao sai:** `eq[2]` luôn là `'='`; biến thứ hai nằm ở `eq[3]`. **Đúng:** `eq[0]` và `eq[3]`.

### Lỗi 6 (bonus): chỉ kiểm hai biến có `==` trực tiếp

In [31]:
eqs = ["a==b", "b==c", "a!=c"]
print("bug:", bug_always_true_without_eq(eqs), "| đúng:", Solution().equationsPossible(eqs))
assert bug_always_true_without_eq(eqs) is True

bug: True | đúng: False


**Vì sao sai:** cùng lỗi bắc cầu: tập kề trực tiếp không phải lớp tương đương.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "quan hệ bằng nhau lan truyền, kèm ràng buộc khác nhau" → **Union-Find** (gộp lớp), kiểm ràng buộc sau khi gộp xong.

**Mẹo:**

1. Hai lượt: gộp `==` rồi kiểm `!=`.
2. `x!=x` tự bị bắt (cùng gốc).
3. `eq[0]`, `eq[3]` là hai biến.
4. 26 biến ⇒ mảng cố định thay vì `dict`.
5. Chứng minh bằng "gán số khác nhau cho mỗi lớp".
6. Oracle: thử mọi phép gán với giá trị `{0..m−1}`.

**Họ bài cùng pattern:** Evaluate Division (399), Number of Provinces (547), Accounts Merge (721), Redundant Connection (684), Smallest String With Swaps (1202), Lexicographically Smallest Equivalent String (1061), Is Graph Bipartite? (785).

## 15. Biến thể

1. **Trả về một phép gán** thoả mãn (hoặc `None`), kiểm chứng bằng cách thay vào mọi phương trình.
2. **Tên biến tổng quát** (chuỗi bất kỳ, không chỉ 26 chữ cái): DSU bằng `dict`.

Cài đặt và kiểm chứng:

In [32]:
def find_assignment(equations):
    parent = {}

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for e in equations:
        if e[1] == "=":
            parent[find(e[0])] = find(e[3])
    for e in equations:
        if e[1] == "!" and find(e[0]) == find(e[3]):
            return None
    roots = {}
    return {v: roots.setdefault(find(v), len(roots)) for v in sorted({c for e in equations for c in (e[0], e[3])})}


def satisfies(equations, val):
    return all((val[e[0]] == val[e[3]]) == (e[1] == "=") for e in equations)


def possible_general(equations):
    """Biến thể 2: phương trình dạng 'tên1==tên2' / 'tên1!=tên2' với tên là chuỗi bất kỳ không chứa '=' hay '!'."""
    parent = {}

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    parsed = []
    for e in equations:
        if "==" in e:
            a, b = e.split("==")
            parsed.append((a, True, b))
        else:
            a, b = e.split("!=")
            parsed.append((a, False, b))
    for a, eq, b in parsed:
        if eq:
            parent[find(a)] = find(b)
    return all(find(a) != find(b) for a, eq, b in parsed if not eq)


rng = random.Random(15)
for _ in range(800):
    eqs = random_equations(rng, rng.randint(1, 8), rng.choice([2, 3, 4, 5]), 0.5)
    ok = possible_brute(eqs)
    asg = find_assignment(eqs)
    assert (asg is not None) == ok
    if asg is not None:
        assert satisfies(eqs, asg)                                                  # (1)
    assert possible_general(eqs) == ok                                              # (2)
named = ["alice==bob", "bob==carol", "dave!=alice", "carol!=dave"]
assert possible_general(named) is True and possible_general(named + ["alice!=carol"]) is False
print("phép gán, tên biến tổng quát: khớp brute")

phép gán, tên biến tổng quát: khớp brute


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi phép gán | `{0..m-1}^m` | `O(m^m·E)` | `O(m)` | oracle |
| 2. Union-Find hai lượt | gộp `==`, kiểm `!=` | `O(E α)` | `O(1)` | **đáp án chính** |
| 3. DFS thành phần | đồ thị các `==` | `O(26+E)` | `O(26+E)` | tường minh |
| 4. Quick-find | mảng nhãn | `O(26E)` | `O(26)` | đơn giản nhất |
| 5. Bao đóng bitmask | Warshall | `O(26²+E)` | `O(26)` | bao đóng tổng quát |

Mọi cách khớp oracle định nghĩa trên `262 143` hệ 3 biến và khớp nhau với 500 phương trình/26 biến.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [33]:
print(Solution().equationsPossible(["a==b", "b!=a"]), Solution().equationsPossible(["a==b", "b==c", "a==c"]),
      Solution().equationsPossible(["a==b", "b!=c", "c==a"]), Solution().equationsPossible(["a!=a"]))

False True False False


### Bài 2 (tính tay)

`equations = ["a==b", "c==d", "b!=c", "d==a"]`. (a) Các lớp tương đương sau khi gộp mọi `==`? (b) Phương trình `!=` có vi phạm không? (c) Đáp án? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện `find` có nén đường đi và hai lượt:

In [34]:
def dsu_todo(equations):
    parent = list(range(26))

    def find(x):
        pass            # TODO 1: tìm gốc của x, nén đường đi (parent[x] = parent[parent[x]])

    for e in equations:
        pass            # TODO 2: nếu e[1] == "=" thì nối gốc của e[0] với gốc của e[3]
    for e in equations:
        pass            # TODO 3: nếu e[1] == "!" và hai biến cùng gốc thì trả False
    return True


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(dsu_todo(e) == w for e, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **DFS thành phần liên thông**:

In [35]:
class MySolution:
    def equationsPossible(self, equations):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().equationsPossible(e) == w for e, w in examples)

### Bài 5 (mini challenge): tên biến tổng quát

Viết `possible_names(equations)` cho phương trình dạng `"tên1==tên2"` / `"tên1!=tên2"` với tên là chuỗi bất kỳ (không chứa `=`/`!`), bằng DSU dùng `dict`. Kiểm bằng `possible_general` ở mục 15 và `possible_brute` khi tên là một chữ cái.

In [36]:
def possible_names(equations):
    pass            # TODO


# Khi làm xong:
# assert possible_names(["alice==bob", "bob==carol", "dave!=alice"]) is True and possible_names(["alice==bob", "bob==carol", "carol!=alice"]) is False

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `==` là bắc cầu; `x!=x` luôn sai.
- Mức 2: `["a==b", "b!=c", "c==a"]`: `a ~ b`, `c ~ a` ⇒ `b ~ c` mâu thuẫn với `b!=c`.
- Mức 3: `False True False False`.

**Bài 2**

- Mức 1: gộp `a==b`, `c==d`, `d==a`.
- Mức 2: `a~b`, `c~d`, `d~a` ⇒ một lớp `{a,b,c,d}`.
- Mức 3: `b!=c` nối hai biến cùng lớp ⇒ vi phạm ⇒ `False`.

**Bài 3**

- Mức 1: `while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]; return x`.
- Mức 2: `parent[find(ord(e[0]) - 97)] = find(ord(e[3]) - 97)`.
- Mức 3: `if find(ord(e[0]) - 97) == find(ord(e[3]) - 97): return False`.

**Bài 4**

- Mức 1: dựng danh sách kề từ các `==`.
- Mức 2: DFS gán nhãn thành phần.
- Mức 3: kiểm `comp[x] != comp[y]` cho mọi `!=`.

**Bài 5**

- Mức 1: tách chuỗi bằng `e.split("==")` / `e.split("!=")`.
- Mức 2: `parent = {}` và `setdefault` trong `find`.
- Mức 3: hai lượt như bài chính.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [37]:
vals = (Solution().equationsPossible(["a==b", "b!=a"]), Solution().equationsPossible(["a==b", "b==c", "a==c"]),
        Solution().equationsPossible(["a==b", "b!=c", "c==a"]), Solution().equationsPossible(["a!=a"]))
print(vals)
assert vals == (False, True, False, False)
for e, v in zip((["a==b", "b!=a"], ["a==b", "b==c", "a==c"], ["a==b", "b!=c", "c==a"], ["a!=a"]), vals):
    assert v == possible_brute(e)
print("Bài 1 OK")

(False, True, False, False)
Bài 1 OK


### Bài 2

In [38]:
eqs = ["a==b", "c==d", "b!=c", "d==a"]
asg_none = find_assignment(eqs)
print("find_assignment:", asg_none)
assert Solution().equationsPossible(eqs) is False and possible_brute(eqs) is False and asg_none is None
# các lớp sau khi gộp mọi ==
sub = find_assignment([e for e in eqs if e[1] == "="])
print("lớp (nhãn theo biến):", sub)
assert len(set(sub.values())) == 1
print("Bài 2 OK: một lớp {a,b,c,d}, b!=c vi phạm => False")

find_assignment: None
lớp (nhãn theo biến): {'a': 0, 'b': 0, 'c': 0, 'd': 0}
Bài 2 OK: một lớp {a,b,c,d}, b!=c vi phạm => False


### Bài 3

In [39]:
def dsu_answer(equations):
    parent = list(range(26))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for e in equations:
        if e[1] == "=":
            parent[find(ord(e[0]) - 97)] = find(ord(e[3]) - 97)
    for e in equations:
        if e[1] == "!" and find(ord(e[0]) - 97) == find(ord(e[3]) - 97):
            return False
    return True


assert all(dsu_answer(e) == w for e, w in examples)
rng = random.Random(3)
for _ in range(500):
    eqs = random_equations(rng, rng.randint(1, 8), rng.choice([2, 3, 4, 5]), 0.5)
    assert dsu_answer(eqs) == possible_brute(eqs)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [40]:
class MySolutionAnswer:
    def equationsPossible(self, equations):
        adj = defaultdict(list)
        for e in equations:
            if e[1] == "=":
                adj[e[0]].append(e[3])
                adj[e[3]].append(e[0])
        comp = {}
        for s in "abcdefghijklmnopqrstuvwxyz":
            if s in comp:
                continue
            comp[s] = s
            stack = [s]
            while stack:
                u = stack.pop()
                for v in adj[u]:
                    if v not in comp:
                        comp[v] = s
                        stack.append(v)
        return all(comp[e[0]] != comp[e[3]] for e in equations if e[1] == "!")


assert all(MySolutionAnswer().equationsPossible(e) == w for e, w in examples)
rng = random.Random(4)
for _ in range(500):
    eqs = random_equations(rng, rng.randint(1, 8), rng.choice([2, 3, 4, 5]), 0.5)
    assert MySolutionAnswer().equationsPossible(eqs) == possible_brute(eqs)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [41]:
def possible_names(equations):
    parent = {}

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    parsed = []
    for e in equations:
        if "==" in e:
            a, b = e.split("==")
            parsed.append((a, True, b))
        else:
            a, b = e.split("!=")
            parsed.append((a, False, b))
    for a, same, b in parsed:
        if same:
            parent[find(a)] = find(b)
    return all(find(a) != find(b) for a, same, b in parsed if not same)


assert possible_names(["alice==bob", "bob==carol", "dave!=alice"]) is True and possible_names(["alice==bob", "bob==carol", "carol!=alice"]) is False
rng = random.Random(5)
for _ in range(500):
    eqs = random_equations(rng, rng.randint(1, 8), rng.choice([2, 3, 4, 5]), 0.5)
    assert possible_names(eqs) == possible_brute(eqs) == possible_general(eqs)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (quan hệ bằng nhau bắc cầu + ràng buộc khác nhau → Union-Find hai lượt):**

```python
class Solution:
    def equationsPossible(self, equations: List[str]) -> bool:
        parent = list(range(26))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        for e in equations:
            if e[1] == "=":
                parent[find(ord(e[0]) - 97)] = find(ord(e[3]) - 97)
        for e in equations:
            if e[1] == "!" and find(ord(e[0]) - 97) == find(ord(e[3]) - 97):
                return False
        return True
```

**Checklist trước khi submit:**

- [ ] Gộp **mọi** `==` trước, rồi mới kiểm `!=` (không một lượt theo thứ tự)?
- [ ] Nối **gốc với gốc** (`parent[find(x)] = find(y)`)?
- [ ] Dùng `eq[0]` và `eq[3]` làm hai biến (không phải `eq[2]`)?
- [ ] `"x!=x"` cho `False` (cùng gốc tự bắt)?
- [ ] Đã thử `["a==b","b!=a"]`, `["a!=c","a==b","b==c"]`, chuỗi bắc cầu dài, `["a!=a"]`?
- [ ] Hiểu vì sao đủ gán mỗi lớp một số riêng biệt?
- [ ] Đúng tên `class Solution`, `equationsPossible(self, equations)`, không in debug.